# 02. THIẾT KẾ CƠ SỞ DỮ LIỆU CLOUD FIRESTORE VÀ SEEDING DỮ LIỆU KHỐI THCS

Tài liệu này thuyết minh chi tiết mô hình dữ liệu (Schema Design) và quy trình Seeding dữ liệu bất biến (Idempotent Seed Process) trong phân hệ Trung học cơ sở (THCS) thuộc hệ thống **Smart Education Center Management System (SmartEdu)**.

## I. KIẾN TRÚC & NGUYÊN TẮC THIẾT KẾ

1. **Single Source of Truth (SSoT)**: Tất cả thông tin nghiệp vụ được lưu trữ tập trung trên Firebase Cloud Firestore, đồng bộ hóa thời gian thực tới tất cả Client thông qua các `onSnapshot` listener.
2. **Idempotency (Tính bất biến)**: Sử dụng các mã định danh tất định (Deterministic IDs) thay vì auto-generated IDs giúp quy trình chạy Seeding nhiều lần vẫn ghi đè khớp bản ghi gốc, tuyệt đối không nhân bản hay làm sai lệch dữ liệu.
3. **Financial Integrity**: Đảm bảo các ràng buộc tài chính nghiêm ngặt: Học phí còn nợ bằng tổng học phí phải nộp trừ đi số tiền thực tế đã thanh toán; các giá trị giao dịch, hóa đơn và chi phí luôn luôn là số dương ($≥ 0$).

## II. THUYẾT MINH CÁC COLLECTION VÀ CHỈ SỐ TOÀN VẸN

### 1. `subjects` (Môn học THCS)
- **Mục đích**: Danh mục môn học cơ bản khối THCS.
- **Cấu trúc mẫu**:
```json
{
  "id": "toan",
  "name": "Toán học",
  "code": "TOAN"
}
```

### 2. `teachers` (Giảng viên cơ hữu)
- **Mục đích**: Danh sách 18 giảng viên môn học (mỗi môn học gồm đúng 3 giáo viên dạy chéo lớp).
- **Cấu trúc mẫu**:
```json
{
  "id": "teacher_toan_1",
  "name": "Trần Quốc Việt",
  "email": "gv.viettoan@smartedu.vn",
  "subjectId": "toan",
  "department": "Tổ Tự Nhiên"
}
```

### 3. `classes` (Lớp học)
- **Mục đích**: 12 lớp học đại diện cho khối 6, 7, 8, 9.
- **Cấu trúc mẫu**:
```json
{
  "id": "class_6A1",
  "name": "6A1",
  "grade": 6,
  "room": "Phòng 101",
  "capacity": 30,
  "schedule": "Thứ 2 - Thứ 4 - Thứ 6 · 08:00",
  "status": "Đang hoạt động",
  "teachers": {
    "toan": "teacher_toan_2",
    "van": "teacher_van_1"
  }
}
```

### 4. `students` (Học viên)
- **Mục đích**: Hồ sơ lý lịch và tình trạng học phí/học tập của 80 học sinh.
- **Cấu trúc mẫu**:
```json
{
  "id": "STU-2026-001",
  "name": "Nguyễn Minh Anh",
  "classId": "class_6A1",
  "className": "6A1",
  "parentId": "PAR-2026-001",
  "parentName": "Nguyễn Thanh Bình",
  "gpa": 8.9,
  "attendanceRate": 96,
  "homeworkCompletion": 92,
  "riskScore": 12,
  "riskLevel": "Low",
  "tuitionPaid": 4500000,
  "tuitionOwed": 0,
  "status": "PAID",
  "financials": {
    "baseFee": 4500000,
    "discount": 0,
    "finalAmount": 4500000,
    "paidAmount": 4500000,
    "tuitionOwed": 0,
    "status": "PAID"
  }
}
```

### 5. `invoices` & `payments` (Hóa đơn và Giao dịch học phí)
- **Invoices**: Ghi nhận hóa đơn phát sinh hàng tháng cho từng lớp.
- **Payments**: Biên lai xác minh dòng tiền nộp học phí.
```json
{
  "id": "INV-2026-001",
  "studentId": "STU-2026-001",
  "studentName": "Nguyễn Minh Anh",
  "className": "6A1",
  "amount": 4500000,
  "discount": 0,
  "finalAmount": 4500000,
  "paidAmount": 4500000,
  "status": "Paid"
}
```

### 6. `scores` (Sổ điểm học bạ)
- **Mục đích**: Ghi nhận kết quả thi thường xuyên, giữa kỳ, cuối kỳ để tính toán điểm trung bình.
```json
{
  "id": "score_STU-2026-001_toan",
  "studentId": "STU-2026-001",
  "studentName": "Nguyễn Minh Anh",
  "classId": "class_6A1",
  "subjectId": "toan",
  "scoreRegular": 8.8,
  "scoreMid": 8.5,
  "scoreFinal": 8.9,
  "average": 8.8,
  "grade": "B",
  "status": "Đã đạt"
}
```

### 7. `expenses` (Chi phí vận hành)
- **Mục đích**: Ghi vết dòng tiền ra của trung tâm (Lương giáo viên, tiền điện nước, thuê mặt bằng, thiết bị, marketing).
```json
{
  "id": "exp_01",
  "category": "Lương giáo viên",
  "description": "Chi lương tháng 7 cho 15 giáo viên",
  "amount": 324000000,
  "expenseDate": "2026-07-31",
  "createdBy": "Lê Hoàng Phong",
  "status": "Đã chi"
}
```

### 8. `homeworks` (Bài tập về nhà)
- **Mục đích**: Lưu trữ bài tập do giáo viên giao và thống kê trạng thái nộp bài.
```json
{
  "id": "HW-1029103",
  "title": "Bài tập Đại số Khối 6 - Chương 1",
  "classId": "class_6A1",
  "className": "6A1",
  "subject": "Toán học",
  "dateAssigned": "2026-08-01",
  "dueDate": "2026-08-08",
  "submittedCount": 18,
  "totalStudents": 20,
  "status": "Đang mở"
}
```

### 9. `auditLogs` (Nhật ký kiểm toán an ninh)
- **Mục đích**: Ghi nhận toàn bộ thao tác nhạy cảm trên hệ thống để phục vụ kiểm tra chéo (Audit Trail).
```json
{
  "id": "AUD-1029104",
  "timestamp": "2026-08-10 19:15:30",
  "actor": "Trần Thị Mai",
  "role": "ACADEMIC_STAFF",
  "action": "GHI DANH HỌC VIÊN",
  "target": "STU-2026-001",
  "ip": "127.0.0.1",
  "status": "Success",
  "details": "Đã ghi danh thành công học viên Nguyễn Minh Anh"
}
```

### 10. `employees` (Nhân sự trung tâm)
- **Mục đích**: Danh sách nhân viên quản trị hành chính trung tâm.
```json
{
  "id": "user_giaovu",
  "name": "Trần Thị Mai",
  "email": "giaovu@smartedu.vn",
  "phone": "0903334444",
  "role": "ACADEMIC_STAFF",
  "status": "Đang hoạt động"
}
```

### 11. `notifications` & `reports` (Thông báo & Báo cáo)
- **notifications**: Quản lý thông báo người dùng theo thời gian thực.
- **reports**: Lưu trữ danh mục báo cáo vận hành định kỳ.

## III. ĐÁNH GIÁ CHẤT LƯỢNG SEEDING & ĐỒNG BỘ

1. **Idempotency**: Các tiến trình seed xóa bản ghi trùng ID và tạo mới ghi đè bằng `setDoc(doc(db, col, id), data)`. Đảm bảo chạy lại không sinh trùng lặp.
2. **Real-time Synchronization**: Thay đổi dữ liệu tức thời phản hồi trên Client thông qua WebSocket kết nối trực tiếp đến Firestore, không cần người dùng thao tác Reload trang.